# DEH 30-Day PySpark Challenge
## Day 11 — String Functions

**Instructions:**
1. Click `File → Save a copy in Drive` before editing
2. Run each cell in order using `Shift + Enter`
3. Complete the assignment cells at the bottom

---

In [ ]:
!pip install pyspark --quiet

In [ ]:
import os
os.environ['AWS_ACCESS_KEY_ID']     = 'your-access-key-here'
os.environ['AWS_SECRET_ACCESS_KEY'] = 'your-secret-key-here'
os.environ['AWS_DEFAULT_REGION']    = 'us-east-1'
BUCKET = 'deh-pyspark-challenge-your-account-id'
print('Credentials set.')

In [ ]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType

spark = (
    SparkSession.builder
    .appName("DEH-PySpark-Challenge")
    .config("spark.jars.packages",
            "org.apache.hadoop:hadoop-aws:3.4.0,com.amazonaws:aws-java-sdk-bundle:1.12.367")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider",
            "com.amazonaws.auth.EnvironmentVariableCredentialsProvider")
    .getOrCreate()
)
print(f"Spark version: {spark.version}")

In [ ]:
customers_schema = StructType([
    StructField("customer_id", StringType(), True),
    StructField("first_name",  StringType(), True),
    StructField("last_name",   StringType(), True),
    StructField("email",       StringType(), True),
    StructField("city",        StringType(), True),
    StructField("state",       StringType(), True),
    StructField("country",     StringType(), True),
    StructField("signup_date", DateType(),   True),
    StructField("segment",     StringType(), True)
])

orders_schema = StructType([
    StructField("order_id",       StringType(),  True),
    StructField("customer_id",    StringType(),  True),
    StructField("product_id",     StringType(),  True),
    StructField("order_date",     DateType(),    True),
    StructField("quantity",       IntegerType(), True),
    StructField("unit_price",     DoubleType(),  True),
    StructField("discount_pct",   IntegerType(), True),
    StructField("status",         StringType(),  True),
    StructField("payment_method", StringType(),  True),
    StructField("region",         StringType(),  True)
])

customers_df = spark.read.option("header", "true").schema(customers_schema).csv(f"s3a://{BUCKET}/data/customers.csv")
orders_df    = spark.read.option("header", "true").schema(orders_schema).csv(f"s3a://{BUCKET}/data/orders.csv")

print(f"Customers: {customers_df.count()} | Orders: {orders_df.count()}")

## Step 5 — Case functions: upper, lower, initcap

In [ ]:
customers_df.select(
    F.col("first_name"),
    F.upper(F.col("first_name")).alias("upper_name"),
    F.lower(F.col("first_name")).alias("lower_name"),
    F.initcap(F.col("first_name")).alias("title_name")
).show(5)

## Step 6 — trim() and length()

In [ ]:
messy_df = spark.createDataFrame(
    [("  James  ",), ("Maria   ",), ("  Robert",)],
    ["name"]
)

messy_df.select(
    F.col("name"),
    F.trim(F.col("name")).alias("trimmed"),
    F.length(F.col("name")).alias("length"),
    F.length(F.trim(F.col("name"))).alias("trimmed_length")
).show()

## Step 7 — concat_ws() and split()

In [ ]:
# concat_ws — join with separator
customers_df.select(
    F.concat_ws(" ", F.col("first_name"), F.col("last_name")).alias("full_name"),
    F.concat_ws(", ", F.col("city"), F.col("state"), F.col("country")).alias("full_address")
).show(5)

In [ ]:
# split() — split on a pattern into an array
customers_df.select(
    F.col("email"),
    F.split(F.col("email"), "@")[0].alias("username"),
    F.split(F.col("email"), "@")[1].alias("domain")
).show(5)

## Step 8 — substring() and regexp_replace()

In [ ]:
# substring(col, start_pos, length) — positions start at 1
orders_df.select(
    F.col("order_id"),
    F.substring(F.col("order_id"), 1, 1).alias("prefix"),
    F.substring(F.col("order_id"), 2, 4).alias("number")
).show(5)

In [ ]:
# regexp_replace(col, pattern, replacement)
orders_df.select(
    F.col("order_id"),
    F.regexp_replace(F.col("order_id"), "[0-9]", "").alias("letters_only"),
    F.regexp_replace(F.col("order_id"), "O", "ORD-").alias("reformatted")
).show(5)

## Step 9 — contains, startswith, endswith

In [ ]:
# Filtering with string matching
orders_df.filter(F.col("payment_method").endswith("Card")).show(3)

# Creating a flag column
orders_df.select(
    F.col("order_id"),
    F.col("payment_method"),
    F.col("payment_method").contains("Card").alias("is_card_payment")
).show(5)

---
## Assignment — Day 11

---

### Task 1
From `customers.csv`, create a `full_name` column combining `first_name` and `last_name` using `concat_ws()`.  
Also create a `location` column combining `city`, `state`, and `country` separated by commas.

In [ ]:
# Task 1 — Write your code here


### Task 2
From `customers.csv`, split the `email` column on `@` to extract `username` and `domain` as separate columns.

In [ ]:
# Task 2 — Write your code here


### Task 3
From `orders.csv`, use `regexp_replace()` to reformat `order_id` — replace the leading `O` with `ORD-`.  
Show original and reformatted values side by side.

In [ ]:
# Task 3 — Write your code here


### Task 4
From `orders.csv`, add a boolean column `is_card_payment` using `contains("Card")` on `payment_method`.  
How many orders were paid by card?

In [ ]:
# Task 4 — Write your code here


---
*DEH 30-Day PySpark Challenge · Data Engineering Hub · RADE Program*